# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import numpy as np
import xarray as xr

psp.del_data('*')

time_range_full = ['2017-11-15/16:00:00', '2017-11-15/17:00:00']
ergpy.lepi(time_range_full, datatype='3dflux', get_support_data=True, version='v03_00')

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:10:00', '2017-11-15/16:27:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = psp.get_data('erg_lepi_l2_3dflux_FPDU', xarray=True).sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

print(flux_data_arrays)

fidu_angle_dict     = psp.get_data('erg_lepi_l2_3dflux_FIDU_Angle_sga', xarray=True)
fidu_angle  = fidu_angle_dict.astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
# AZ_deg_mid =  [ 78.75  56.25  33.75  11.25 -11.25 -33.75 -56.25 -78.75]

theta_sga = AZ_deg_mid                  # (C,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)

angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)

# theta_sga, varphi_sga -> Vx_sga, Vy_sga, Vz_sga
vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.sin(np.radians(angle_sga_time[:, :, 0]))

v_unit_vector_sga_energy_channel_list = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1)

    for channel_i in range(channel_num):
        v_unit_vector_sga_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            vector_sga_time[:, channel_i, :],
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sga_{energy_i}_{channel_i}'
        )
        dot_ = (v_unit_vector_sga_energy_channel_list[energy_i, channel_i] * v_unit_vector_sga_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sga_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

In [ ]:
import pyspedas as psp
import xarray as xr

v_unit_vector_sgi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        psp.store_data(f'vector_sga_{energy_i}_{channel_i}', data={'x': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].time, 'y': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].values})
        # SGI座標系に変換
        psp.projects.erg.sga2sgi(name_in=f'vector_sga_{energy_i}_{channel_i}', name_out=f'vector_sgi_{energy_i}_{channel_i}')
        _data   = psp.get_data(f'vector_sgi_{energy_i}_{channel_i}', xarray=True).rename({"v_dim": "xyz"})
        _data_unit  = _data
        v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sgi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        _dot    = (v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] * v_unit_vector_sgi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sgi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(_dot), np.nanmax(_dot), np.nanmean(_dot))

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        # dsi座標系に変換
        psp.projects.erg.sgi2dsi(name_in=f'vector_sgi_{energy_i}_{channel_i}', name_out=f'vector_dsi_{energy_i}_{channel_i}')
        _data   = psp.get_data(f'vector_dsi_{energy_i}_{channel_i}', xarray=True).rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_dsi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        dot_    = (v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] * v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

# 背景磁場ベクトルの決定

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import xarray as xr
import numpy as np

ergpy.mgf(trange=time_range_full, level='l2', datatype='256hz', coord='dsi', version='v03.03')
ergpy.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', version='v03.03')

B_256Hz = psp.get_data('erg_mgf_l2_mag_256hz_dsi', xarray=True).rename({"v_dim": "xyz"})
B_8sec  = psp.get_data('erg_mgf_l2_mag_8sec_dsi', xarray=True).rename({"v_dim": "xyz"})

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
B0_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        B_256Hz_interp       = B_256Hz.interp(time=v_unit_vector_dsi_energy_channel_list[energy_i, channel_i].time, method='linear')
        dt_B_256Hz_interp    = (B_256Hz_interp.time[1] - B_256Hz_interp.time[0]) / np.timedelta64(1, 's')
        B_background        = B_256Hz_interp.rolling(time=int(background_time_sec/dt_B_256Hz_interp), center=True).mean('time')

        B0_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            B_background.data,
            dims=['time', 'xyz'],
            coords={
                'time': B_background.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'B0_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'B0_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', B0_vector_dsi_energy_channel_list[energy_i, channel_i])

# 対象とする垂直磁場成分を取得

In [ ]:
t_B_8sec    = B_8sec.time
dt_B_8sec = (t_B_8sec[2] - t_B_8sec[1]) / np.timedelta64(1, 's')
B_background    = B_8sec.rolling(time=int(background_time_sec/dt_B_8sec), center=True).mean('time')

B_background_256Hz  = B_background.interp(time=B_256Hz.time, method='linear')
B_256Hz_perturb     = B_256Hz - B_background_256Hz
B_256Hz_perp = B_256Hz_perturb - (B_256Hz_perturb * B_background_256Hz).sum(dim='xyz') / (B_background_256Hz * B_background_256Hz).sum(dim='xyz') * B_background_256Hz

In [ ]:
import numpy as np
import xarray as xr
from scipy.signal import spectrogram, detrend, get_window

def spectro_to_xarray(da, fs=256.0, comp=2, nperseg=4096, noverlap=None, window='hann'):
    """
    da: DataArray(time, xyz) 例: B_256Hz
    comp: 0=Bx, 1=By, 2=Bz
    返り値: Dataset {Sxx_<comp>} with coords time(freq center), freq
    """
    if noverlap is None:
        noverlap = nperseg // 2

    x = detrend(da.isel(xyz=comp).values, type='linear')
    win = get_window(window, nperseg)

    f, t, Sxx = spectrogram(x, fs=fs, window=win, nperseg=nperseg,
                            noverlap=noverlap, detrend='linear',
                            scaling='density', mode='psd')   # Sxx: (F, T)

    # spectrogram の t は開始からの秒。絶対時刻へ変換（窓中心時刻）
    t0 = da.time.values[0].astype('datetime64[ns]')
    t_abs = t0 + (np.rint(t * 1e9).astype('int64')).astype('timedelta64[ns]')

    varname = {0:'Bx', 1:'By', 2:'Bz'}[comp]
    ds = xr.Dataset(
        data_vars={f'Sxx_{varname}': (('time', 'freq'), Sxx.T)},   # (T,F)
        coords={'time': t_abs, 'freq': f},
        attrs=dict(fs=fs, nperseg=nperseg, noverlap=noverlap, window=window,
                   input_units='nT', psd_units='nT^2/Hz',
                   method='scipy.signal.spectrogram')
    )
    return ds

In [ ]:
ds_spec_B_256Hz_z = spectro_to_xarray(B_256Hz, fs=256.0, comp=2, nperseg=4096)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import spectrogram, detrend, get_window
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:16:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')

# ---- 描画 ----
plt.figure(figsize=(8, 3))
TIME, FREC = np.meshgrid(ds_spec_B_256Hz_z.time, ds_spec_B_256Hz_z.freq)
plt.pcolormesh(TIME, FREC, ds_spec_B_256Hz_z.Sxx_Bz.T, shading='auto', cmap='turbo',
               norm=LogNorm(vmin=1E-4, vmax=1E2))
plt.ylim(0.4, 1.0)
plt.xlim(time_ax_range_min, time_ax_range_max)
plt.xlabel('Time')
plt.ylabel('Frequency [Hz]')
plt.colorbar(label=r'PSD [nT$^2$/Hz]')
plt.title('Bz spectrogram (256 Hz sampling)')
plt.minorticks_on()
plt.grid(which='both', alpha=0.5, linestyle=':')
plt.tight_layout()
plt.show()


In [ ]:
# 0.6 Hzから0.75 Hzの範囲のみを通すband-passフィルタ
lowcut = 0.45
highcut = 0.75

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 256.                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）

# btypeを'bandpass'に設定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

# ------------------ インパルス応答 (変更なし) ------------------
n = 4096
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-pass filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)|')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(lowcut, highcut, color='gray', alpha=0.3, label=f'{lowcut:.2f}-{highcut:.2f} Hz')
axs[1].axvline(lowcut, color='red', lw=1)
axs[1].axvline(highcut, color='red', lw=1)

axs[1].set_title('Magnitude Response (Band-pass filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(3E-1, 2E0)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# フィルタパラメータ
fs = 256.                  # サンプリング周波数 [Hz]
order = 4                 # フィルタの次数
window_sec = background_time_sec        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

B_256Hz_perp_bandpass = np.zeros(B_256Hz_perp.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    B_256Hz_perp_bandpass[:, i] = apply_filter_segmented(B_256Hz_perp.data[:, i], sos)
da_B_256Hz_perp_bandpass = xr.DataArray(
    B_256Hz_perp_bandpass,
    dims=B_256Hz_perp.dims,
    coords=B_256Hz_perp.coords,
    name='B_256Hz_perp_bandpass'
)

da_B_256Hz_perp_bandpass_amp = np.sqrt((da_B_256Hz_perp_bandpass * da_B_256Hz_perp_bandpass).sum(dim='xyz'))

In [ ]:
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:16:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')

# plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4), sharex=True)
ax.plot(da_B_256Hz_perp_bandpass_amp.time, da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax.set_ylabel('[nT]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

# v_unitとB0、B_perpとのなす角を求めて、pitch angleとzeta angleをfluxデータに付与

In [ ]:
def ensure_xyz_coord(da):
    if 'xyz' in da.dims and 'xyz' not in da.coords:
        da = da.assign_coords(xyz=['x','y','z'])
    return da

flux_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        v_unit  = v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]
        B0 = B0_vector_dsi_energy_channel_list[energy_i, channel_i].interp(time=v_unit.time)
        Bperp = da_B_256Hz_perp_bandpass.interp(time=v_unit.time)
        Bperp = Bperp - (Bperp * B0).sum(dim='xyz') / (B0 * B0).sum(dim='xyz') * B0

        v_unit  = ensure_xyz_coord(v_unit)
        B0      = ensure_xyz_coord(B0)
        Bperp   = ensure_xyz_coord(Bperp)

        dot_vB0 = (v_unit * B0).sum(dim='xyz')
        B0_2    = (B0 * B0).sum(dim='xyz')
        alpha = np.arccos(dot_vB0 / np.sqrt(B0_2))

        v_perp = v_unit - dot_vB0 / B0_2 * B0
        cross = xr.apply_ufunc(np.cross, Bperp, v_perp,
                               input_core_dims=[['xyz'], ['xyz']],
                               output_core_dims=[['xyz']], vectorize=True)
        Bperp_2     = (Bperp * Bperp).sum(dim='xyz')
        sin_zeta    = (cross * B0).sum(dim='xyz') / np.sqrt(B0_2 * Bperp_2)
        cos_zeta    = (v_perp * Bperp).sum(dim='xyz') / np.sqrt(Bperp_2)
        zeta = np.atan2(sin_zeta, cos_zeta)
        
        # 時間を統一（zeta基準）
        t = zeta['time']
        
        # flux の time を zeta に合わせる（必要なら補間）
        flux_ch = xr.DataArray(
            flux_data_arrays[energy_i][:, channel_i],
            coords={'time': flux_data_arrays[energy_i].coords['time']},  # ここは実データのtimeに合わせる
            dims=('time',)
        ).interp(time=t)
        
        da = xr.concat(
            [
                flux_ch.rename('differential_number_flux_keV'),
                np.rad2deg(alpha).rename('pitch_angle_deg'),
                (np.rad2deg(zeta) % 360.0).rename('zeta_angle_deg')
            ],
            dim='variable'
        ).assign_coords(variable=['differential_number_flux_keV','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        flux_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(flux_pitch_zeta_data_list[energy_i, channel_i])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        flux_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    flux_all = np.concatenate(flux_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0) & (alpha_all > 125) & (alpha_all < 145)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        mask = (alpha >= 125) & (alpha <= 145) & (flux > 0)
        t_mask  = t[mask]
        flux_mask   = flux[mask]
        flux_mask_ratio = flux_mask
        alpha_mask  = alpha[mask]
        zeta_mask   = zeta[mask]
        ax0.scatter(t_mask, alpha_mask, c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t_mask, zeta_mask,  c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    plt.tight_layout()
    plt.show()


# Differential Number Flux $J$ をCount Number $C$に変換

In [ ]:
sampling_time   = 0.015625  # [sec]

In [ ]:
def G_Factor_func(energy):
    return  (1.52 - 0.108 * np.log10(energy)) * 1E-3  # [cm^-2 str keV keV^-1 channel^-1]

G_Factor_ax = G_Factor_func(energy_ax)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 5), sharex=True)
ax.plot(energy_ax, G_Factor_ax, lw=2, c='red')
ax.set_xlabel(r'Energy per charge [$\mathrm{keV/q}$]')
ax.set_ylabel(r'Geometric factor ($G_{\mathrm{ESA}}$)' + '\n' + r'[$\mathrm{cm}^{2} \, \mathrm{str} \, \mathrm{keV} \, \mathrm{keV}^{-1} \, \mathrm{channel}^{-1}$]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xscale('log')
ax.set_xlim(np.nanmin(energy_ax), np.nanmax(energy_ax))
ax.set_ylim(0.00135, 0.0018)
ax.axvline(0.01, c='b', linestyle=':', lw=2)
ax.axhline(0.00170, c='b', linestyle=':', lw=2)
ax.axvline(12, c='orange', linestyle=':', lw=2)
ax.axhline(0.00140, c='orange', linestyle=':', lw=2)
plt.tight_layout()
plt.show()

In [ ]:
Efficiency  = 1.0 #0.7   # Asamura et al. (2018)でのdetection efficiency of STOP signalsを仮採用

In [ ]:
count_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        data_ = flux_pitch_zeta_data_list[energy_i, channel_i]
        G_Factor_   = G_Factor_func(energy_ax[energy_i])

        count_ch = sampling_time * Efficiency * G_Factor_ * energy_ax[energy_i] * data_[:, 0]   # C = τ * ε * G * E * J

        da = xr.concat(
            [
                count_ch.rename('count_number'),
                data_[:, 1],
                data_[:, 2]
            ],
            dim='variable'
        ).assign_coords(variable=['count_number','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        count_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(energy_ax[energy_i])
        print(count_pitch_zeta_data_list[energy_i, channel_i][900:930, :])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    #if (energy_i == 0) | (energy_i > 12):
    if (energy_i != 5):
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    count_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = count_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='count_number').values
        alpha = d.sel(variable='pitch_angle_deg').values
        count_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    count_all = np.concatenate(count_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(count_all) & (alpha_all >= 125) & (alpha_all <= 145) #& (count_all > 1)
    if not mask.any():
        continue
    vmin, vmax = count_all[mask].min(), count_all[mask].max()
    vmax = np.ceil(vmax)
    #vmin = 1
    vmin    = np.nanmax([vmin, vmax*1E-2])
    #norm    = mcolors.Normalize(vmin=0, vmax=vmax)
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = count_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        count = d.sel(variable='count_number').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values
        mask = np.isfinite(count) & (alpha > 125) & (alpha < 145) #& (count > 1)
        t   = t[mask]
        count   = count[mask]
        alpha   = alpha[mask]
        zeta    = zeta[mask]
        ax0.scatter(t, alpha, c=count, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=count, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i count number (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 10))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Count Number')
    plt.colorbar(sm, ax=ax1, label='Count Number')

    plt.tight_layout()
    plt.show()


# 電場ベクトルの取得

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import xarray as xr
import numpy as np

ergpy.pwe_efd(trange=time_range_full, level='l2', datatype='256hz', coord='dsi')

In [ ]:
Ex_256Hz_dsi = psp.get_data('erg_pwe_efd_l2_E256Hz_dsi_Ex_waveform', xarray=True)
Ey_256Hz_dsi = psp.get_data('erg_pwe_efd_l2_E256Hz_dsi_Ey_waveform', xarray=True)

Ex_256Hz_dsi_interp  = Ex_256Hz_dsi.interp(time=B_256Hz_perturb.time)
Ey_256Hz_dsi_interp  = Ey_256Hz_dsi.interp(time=B_256Hz_perturb.time)

#Ez_256Hz_dsi_interp  = - (Ex_256Hz_dsi_interp * B_256Hz_perturb[:, 0] + Ey_256Hz_dsi_interp * B_256Hz_perturb[:, 1]) / B_256Hz_perturb[:, 2]

Ez_256Hz_dsi_interp  = xr.where(np.abs(B_background_256Hz[:, 2]) > 1E-5, -(Ex_256Hz_dsi_interp * B_background_256Hz[:, 0] + Ey_256Hz_dsi_interp * B_background_256Hz[:, 1]) / B_background_256Hz[:, 2], np.nan)

E_256Hz  = xr.concat(
    [
        Ex_256Hz_dsi_interp,
        Ey_256Hz_dsi_interp,
        Ez_256Hz_dsi_interp
    ],
    dim='xyz'
).transpose('time', 'xyz')

In [ ]:
E_256Hz_perp    = E_256Hz - (E_256Hz * B_background_256Hz).sum(dim='xyz') / (B_background_256Hz * B_background_256Hz).sum(dim='xyz') * B_background_256Hz

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# フィルタパラメータ
fs = 256.                  # サンプリング周波数 [Hz]
order = 4                 # フィルタの次数
window_sec = background_time_sec        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

E_256Hz_perp_bandpass = np.zeros(E_256Hz_perp.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    E_256Hz_perp_bandpass[:, i] = apply_filter_segmented(E_256Hz_perp.data[:, i], sos)
da_E_256Hz_perp_bandpass = xr.DataArray(
    E_256Hz_perp_bandpass,
    dims=E_256Hz_perp.dims,
    coords=E_256Hz_perp.coords,
    name='E_256Hz_perp_bandpass'
)

da_E_256Hz_perp_bandpass_amp = np.sqrt((da_E_256Hz_perp_bandpass * da_E_256Hz_perp_bandpass).sum(dim='xyz'))

In [ ]:
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:15:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')
fig, ax = plt.subplots(1, 1, figsize=(10, 4), sharex=True)
ax.plot(da_E_256Hz_perp_bandpass_amp.time, da_E_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax.set_ylabel('[mV/m]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
#ax.set_ylim(0, 1000)
plt.tight_layout()
plt.show()

# 確認する物理量の分布を出力
For $W_{\mathrm{Eint}}$ and $W_{\mathrm{Bint}}$, we obtain these quantities
```math
\frac{q E_{\mathrm{w}i} C_{i}}{\varepsilon G_{i}} \sin^{2} \alpha \, [\mathrm{eV \, m^{-3} \, rad^{-2}}] \quad \mathrm{and} \quad \frac{q B_{\mathrm{w}i} C_{i}}{\varepsilon G_{i}} \sin^{2} \alpha \, [\mathrm{eV \, s \, m^{-4} \, rad^{-2}}].
```

In [ ]:
modified_count_E_pitch_zeta_data_list = {}
modified_count_B_pitch_zeta_data_list = {}

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        data_       = count_pitch_zeta_data_list[energy_i, channel_i]

        count_      = data_[:, 0]
        alpha_      = np.deg2rad(data_[:, 1])   # [rad]
        zeta_       = np.deg2rad(data_[:, 2])   # [rad]
        energy_     = energy_ax[energy_i]       # [keV]
        G_Factor_   = G_Factor_func(energy_)    # [cm^2 str keV keV^-1]

        Eperp_      = da_E_256Hz_perp_bandpass.interp(time=count_.time) * 1E-3  # [V/m]
        Eperp_2_    = (Eperp_ * Eperp_).sum(dim='xyz')
        Bperp_      = da_B_256Hz_perp_bandpass.interp(time=count_.time) * 1E-9  # [T]
        Bperp_2_    = (Bperp_ * Bperp_).sum(dim='xyz')

        count_E_    = 1E4 * np.sqrt(Eperp_2_) * count_ / Efficiency / G_Factor_ * np.sin(alpha_.data)**2E0  # [eV m^-3 rad^-2]
        count_B_    = 1E4 * np.sqrt(Bperp_2_) * count_ / Efficiency / G_Factor_ * np.sin(alpha_.data)**2E0  # [eV s m^-4 rad^-2]

        da_E_ = xr.concat(
            [
                count_E_,
                data_[:, 1],
                data_[:, 2]
            ],
            dim='variable'
        ).assign_coords(variable=['modified_count_E', 'pitch_angle_deg','zeta_angle_deg']).transpose('time','variable')

        da_B_ = xr.concat(
            [
                count_B_,
                data_[:, 1],
                data_[:, 2]
            ],
            dim='variable'
        ).assign_coords(variable=['modified_count_B', 'pitch_angle_deg','zeta_angle_deg']).transpose('time','variable')

        modified_count_E_pitch_zeta_data_list[energy_i, channel_i] = da_E_.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(modified_count_E_pitch_zeta_data_list[energy_i, channel_i][900:910, :])

        modified_count_B_pitch_zeta_data_list[energy_i, channel_i] = da_B_.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(modified_count_B_pitch_zeta_data_list[energy_i, channel_i][900:910, :])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:15:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')

for energy_i in range(energy_num):
    if energy_i != 2:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    count_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = modified_count_E_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='modified_count_E').values
        alpha = d.sel(variable='pitch_angle_deg').values
        count_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    count_all = np.concatenate(count_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(count_all) & (alpha_all > 125) & (alpha_all < 145)
    if not mask.any():
        continue
    #vmin, vmax = count_all[mask].min(), count_all[mask].max()
    #vmax_plus = np.nanmax([np.abs(vmin), np.abs(vmax)])
    #norm = mcolors.SymLogNorm(vmin= - vmax_plus, vmax=vmax_plus, linthresh=vmax_plus*1E-3)
    #cmap = 'bwr'
    vmin, vmax = count_all[mask].min(), count_all[mask].max()
    vmax_plus = np.nanmax([np.abs(vmin), np.abs(vmax)])
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax_plus)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        d = modified_count_E_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        count = d.sel(variable='modified_count_E').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values
        mask = np.isfinite(count) & (alpha > 125) & (alpha < 145)
        t   = t[mask]
        count   = count[mask]
        alpha   = alpha[mask]
        zeta    = zeta[mask]
        ax0.scatter(t, alpha, c=count, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=count, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i modified count number for E (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 10))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Modified count ' + r'[$\mathrm{eV} \, \mathrm{m}^{-3} \, \mathrm{str}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Modified count ' + r'[$\mathrm{eV} \, \mathrm{m}^{-3} \, \mathrm{str}^{-1}$]')

    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:15:00')
time_ax_range_max = np.datetime64('2017-11-15T16:25:00')

for energy_i in range(energy_num):
    if energy_i != 2:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    count_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = modified_count_B_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='modified_count_B').values
        alpha = d.sel(variable='pitch_angle_deg').values
        count_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    count_all = np.concatenate(count_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(count_all) & (alpha_all > 125) & (alpha_all < 145)
    if not mask.any():
        continue
    #vmin, vmax = count_all[mask].min(), count_all[mask].max()
    #vmax_plus = np.nanmax([np.abs(vmin), np.abs(vmax)])
    #norm = mcolors.SymLogNorm(vmin= - vmax_plus, vmax=vmax_plus, linthresh=vmax_plus*1E-3)
    #cmap = 'bwr'
    vmin, vmax = count_all[mask].min(), count_all[mask].max()
    vmax_plus = np.nanmax([np.abs(vmin), np.abs(vmax)])
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax_plus)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        d = modified_count_B_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        count = d.sel(variable='modified_count_B').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values
        mask = np.isfinite(count) & (alpha > 125) & (alpha < 145)
        t   = t[mask]
        count   = count[mask]
        alpha   = alpha[mask]
        zeta    = zeta[mask]
        ax0.scatter(t, alpha, c=count, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=count, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i modified count number for B (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 10))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Modified count ' + r'[$\mathrm{eV} \, \mathrm{s} \, \mathrm{m}^{-4} \, \mathrm{str}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Modified count ' + r'[$\mathrm{eV} \, \mathrm{s} \, \mathrm{m}^{-4} \, \mathrm{str}^{-1}$]')

    plt.tight_layout()
    plt.show()


In [ ]:
energy_center   = energy_ax[1:]

log_center = np.log10(energy_center)
log_edges = np.zeros(len(energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

energy_grid = 10**log_edges

energy_width = np.zeros(len(energy_center))
energy_width = energy_grid[0:-1] - energy_grid[1:]

energy_width_center = energy_width / energy_center
energy_width

In [ ]:
import numpy as np
import xarray as xr

def make_weighted_zeta_counts_for_energy_periodic_avg_sin(
    count_pitch_zeta_data_list,
    count_variable_name,
    energy_i,
    channel_num,
    time_start,
    time_end,
    T_integrate=60.0,      # [s]
    step_sec=4.0,          # [s]
    alpha_min=125.0,
    alpha_max=145.0,
    zeta_bin_width=30.0    # [deg]
):
    """
    1エネルギーチャンネルについて、ピッチ角 α∈[alpha_min, alpha_max] の
    カウントを ζ ビン(0,30,...,330)に重み付きで分配し、
    各ビンごとに [Σ(C * weight) / Σ(weight)] * T_integrate を返す（ζ は周期境界）。
    """

    # ζ グリッド（ビン中心）
    zeta_centers = np.arange(0., 360.0, zeta_bin_width)  # 0,30,...,330
    n_bins       = zeta_centers.size

    # 時間中心 tc のリスト
    time_start = np.datetime64(time_start)
    time_end   = np.datetime64(time_end)
    tc_list = np.arange(
        time_start,
        time_end + np.timedelta64(1, "s"),
        np.timedelta64(int(step_sec), "s")
    ).astype("datetime64[ns]")

    # 積分時間の半分
    T_half = np.timedelta64(int(T_integrate / 2), "s")

    # 出力配列 (tc, zeta_bin)
    num_tc_zeta = np.zeros((tc_list.size, n_bins), dtype=float)  # Σ(C * weight)
    den_tc_zeta = np.zeros((tc_list.size, n_bins), dtype=float)  # Σ(weight)

    for itc, tc in enumerate(tc_list):
        t0 = tc - T_half
        t1 = tc + T_half

        for ch in range(channel_num):
            d = count_pitch_zeta_data_list[energy_i, ch].sel(time=slice(t0, t1))
            if d.time.size == 0:
                continue

            count = d.sel(variable=count_variable_name).values
            alpha = d.sel(variable="pitch_angle_deg").values
            zeta  = d.sel(variable="zeta_angle_deg").values

            # 有効データ & α 範囲
            mask = (
                np.isfinite(count)
                & np.isfinite(alpha)
                & np.isfinite(zeta)
                & (alpha >= alpha_min)
                & (alpha <= alpha_max)
                & (count >= 1.)
            )
            if not mask.any():
                continue

            c = count[mask].ravel()
            a = alpha[mask].ravel()
            z = zeta[mask].ravel()

            # ---- 周期境界つき ζ 線形分配 ----
            # [0, 360) に折りたたみ
            z_mod = np.mod(z, 360.0)

            # 左のビン中心 index (0..n_bins-1)
            i0 = np.floor(z_mod / zeta_bin_width).astype(int)
            i0 = np.clip(i0, 0, n_bins - 1)

            # 右のビン中心（周期境界）
            i1 = (i0 + 1) % n_bins

            # 左の中心角 ζ0
            z0 = zeta_centers[i0]
            z1 = zeta_centers[i1]
            dz = zeta_bin_width  # Δζ は一定

            # 線形重み
            w1 = ((z_mod - z0) / dz)
            w0 = (1.0 - w1)

            # 分子: Σ(C * weight)
            np.add.at(num_tc_zeta[itc], i0, c * w0 * np.sin(np.deg2rad(z0)) * np.deg2rad(dz))
            np.add.at(num_tc_zeta[itc], i1, c * w1 * np.sin(np.deg2rad(z1)) * np.deg2rad(dz))

            # 分母: Σ(weight)
            np.add.at(den_tc_zeta[itc], i0, w0)
            np.add.at(den_tc_zeta[itc], i1, w1)
            # ---- ここまで ----

    # 重み付き平均 × T_integrate
    with np.errstate(invalid='ignore', divide='ignore'):
        counts_tc_zeta = (num_tc_zeta / den_tc_zeta) * T_integrate

    da = xr.DataArray(
        counts_tc_zeta,
        coords={
            "time_center": tc_list,
            "zeta_center": zeta_centers,
        },
        dims=("time_center", "zeta_center"),
        name="count_weighted_avg",
        attrs={
            "T_integrate_sec": T_integrate,
            "step_sec": step_sec,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "zeta_bin_width_deg": zeta_bin_width,
        },
    )
    return da


In [ ]:
import numpy as np
import xarray as xr

def make_weighted_zeta_counts_for_energy_periodic_avg_cos(
    count_pitch_zeta_data_list,
    count_variable_name,
    energy_i,
    channel_num,
    time_start,
    time_end,
    T_integrate=60.0,      # [s]
    step_sec=4.0,          # [s]
    alpha_min=125.0,
    alpha_max=145.0,
    zeta_bin_width=30.0    # [deg]
):
    """
    1エネルギーチャンネルについて、ピッチ角 α∈[alpha_min, alpha_max] の
    カウントを ζ ビン(0,30,...,330)に重み付きで分配し、
    各ビンごとに [Σ(C * weight) / Σ(weight)] * T_integrate を返す（ζ は周期境界）。
    """

    # ζ グリッド（ビン中心）
    zeta_centers = np.arange(0., 360.0, zeta_bin_width)  # 0,30,...,330
    n_bins       = zeta_centers.size

    # 時間中心 tc のリスト
    time_start = np.datetime64(time_start)
    time_end   = np.datetime64(time_end)
    tc_list = np.arange(
        time_start,
        time_end + np.timedelta64(1, "s"),
        np.timedelta64(int(step_sec), "s")
    ).astype("datetime64[ns]")

    # 積分時間の半分
    T_half = np.timedelta64(int(T_integrate / 2), "s")

    # 出力配列 (tc, zeta_bin)
    num_tc_zeta = np.zeros((tc_list.size, n_bins), dtype=float)  # Σ(C * weight)
    den_tc_zeta = np.zeros((tc_list.size, n_bins), dtype=float)  # Σ(weight)

    for itc, tc in enumerate(tc_list):
        t0 = tc - T_half
        t1 = tc + T_half

        for ch in range(channel_num):
            d = count_pitch_zeta_data_list[energy_i, ch].sel(time=slice(t0, t1))
            if d.time.size == 0:
                continue

            count = d.sel(variable=count_variable_name).values
            alpha = d.sel(variable="pitch_angle_deg").values
            zeta  = d.sel(variable="zeta_angle_deg").values

            # 有効データ & α 範囲
            mask = (
                np.isfinite(count)
                & np.isfinite(alpha)
                & np.isfinite(zeta)
                & (alpha >= alpha_min)
                & (alpha <= alpha_max)
                & (count >= 1.)
            )
            if not mask.any():
                continue

            c = count[mask].ravel()
            a = alpha[mask].ravel()
            z = zeta[mask].ravel()

            # ---- 周期境界つき ζ 線形分配 ----
            # [0, 360) に折りたたみ
            z_mod = np.mod(z, 360.0)

            # 左のビン中心 index (0..n_bins-1)
            i0 = np.floor(z_mod / zeta_bin_width).astype(int)
            i0 = np.clip(i0, 0, n_bins - 1)

            # 右のビン中心（周期境界）
            i1 = (i0 + 1) % n_bins

            # 左の中心角 ζ0
            z0 = zeta_centers[i0]
            z1 = zeta_centers[i1]
            dz = zeta_bin_width  # Δζ は一定

            # 線形重み
            w1 = ((z_mod - z0) / dz)
            w0 = (1.0 - w1)

            # 分子: Σ(C * weight)
            np.add.at(num_tc_zeta[itc], i0, c * w0 * np.cos(np.deg2rad(z0)) * np.deg2rad(dz))
            np.add.at(num_tc_zeta[itc], i1, c * w1 * np.cos(np.deg2rad(z1)) * np.deg2rad(dz))

            # 分母: Σ(weight)
            np.add.at(den_tc_zeta[itc], i0, w0)
            np.add.at(den_tc_zeta[itc], i1, w1)
            # ---- ここまで ----

    # 重み付き平均 × T_integrate
    with np.errstate(invalid='ignore', divide='ignore'):
        counts_tc_zeta = (num_tc_zeta / den_tc_zeta) * T_integrate

    da = xr.DataArray(
        counts_tc_zeta,
        coords={
            "time_center": tc_list,
            "zeta_center": zeta_centers,
        },
        dims=("time_center", "zeta_center"),
        name="count_weighted_avg",
        attrs={
            "T_integrate_sec": T_integrate,
            "step_sec": step_sec,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "zeta_bin_width_deg": zeta_bin_width,
        },
    )
    return da


In [ ]:
WEint_energy_list   = {}
WBint_energy_list   = {}

alpha_min_deg   = 125.0
alpha_max_deg   = 145.0
delta_alpha_rad = np.deg2rad(alpha_max_deg - alpha_min_deg)

for energy_i in range(energy_num):
    if energy_i == 0:
        continue
    da_weighted_E = make_weighted_zeta_counts_for_energy_periodic_avg_sin(
        modified_count_E_pitch_zeta_data_list,
        count_variable_name='modified_count_E',
        energy_i=energy_i,
        channel_num=channel_num,
        time_start=np.datetime64('2017-11-15T16:15:00'),
        time_end=np.datetime64('2017-11-15T16:25:00'),
        T_integrate=60.0,
        step_sec=4.0,
        alpha_min=125.0,
        alpha_max=145.0,
        zeta_bin_width=30.
    )
    #print(da_weighted_E)
    da_WE = xr.DataArray(
        (- da_weighted_E.sum('zeta_center') * delta_alpha_rad).data * energy_width[energy_i - 1] / energy_center[energy_i - 1],
        dims=['time'],
        coords={'time': da_weighted_E.time_center.data,
                'energy_center_keV': energy_center[energy_i - 1],
                'energy_width':      energy_width[energy_i - 1]},
        name=f'W_Eint_per_energy_{energy_i}'
    )
    WEint_energy_list[energy_i] = da_WE # [eV m^-3 s]

    da_weighted_B = make_weighted_zeta_counts_for_energy_periodic_avg_cos(
        modified_count_B_pitch_zeta_data_list,
        count_variable_name='modified_count_B',
        energy_i=energy_i,
        channel_num=channel_num,
        time_start=np.datetime64('2017-11-15T16:15:00'),
        time_end=np.datetime64('2017-11-15T16:25:00'),
        T_integrate=60.0,
        step_sec=4.0,
        alpha_min=125.0,
        alpha_max=145.0,
        zeta_bin_width=30.
    )
    print(da_weighted_B[75:80, :])
    da_WB = xr.DataArray(
        (da_weighted_B.sum('zeta_center') * delta_alpha_rad).data * energy_width[energy_i - 1] / energy_center[energy_i - 1],
        dims=['time'],
        coords={'time': da_weighted_B.time_center.data,
                'energy_center_keV': energy_center[energy_i - 1],
                'energy_width':      energy_width[energy_i - 1]},
        name=f'W_Bint_per_energy_{energy_i}'
    )
    WBint_energy_list[energy_i]  = da_WB   # [eV m^-4 s^2]

In [ ]:
W_Eint_all = xr.concat(list(WEint_energy_list.values()), dim='energy_center_keV')
W_Bint_all = xr.concat(list(WBint_energy_list.values()), dim='energy_center_keV')

print(W_Eint_all[:, 75:80])
print('')
print(W_Bint_all[:, 75:80])

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_W_on_ax(ax, W_all, energy_range=None, time_range=None,
                 cmap="turbo", normalize=True, vlim=None,
                 title=None, ylabel="Energy [keV]", cbar_title=None, cax=None):
    Wa = W_all

    # --- energy 範囲: 降順でも動くようにブール抽出 ---
    if energy_range is not None:
        el, eh = energy_range
        ec = Wa.coords["energy_center_keV"].values
        mask = (ec >= min(el, eh)) & (ec <= max(el, eh))
        Wa = Wa.isel(energy_center_keV=mask)

    # --- time 範囲 ---
    if time_range is not None:
        t0, t1 = np.datetime64(time_range[0]), np.datetime64(time_range[1])
        Wa = Wa.sel(time=slice(t0, t1))

    if Wa.size == 0:
        raise ValueError("指定範囲内にデータが存在しない")

    # --- energy 昇順に並べ替え ---
    e = Wa.energy_center_keV.values
    order = np.argsort(e)
    e = e[order]
    Wa = Wa.isel(energy_center_keV=order)

    # --- energy エッジ（対数） ---
    loge = np.log10(e)
    loge_edge = np.empty(e.size + 1)
    loge_edge[1:-1] = 0.5 * (loge[:-1] + loge[1:])
    loge_edge[0]    = loge[0]  + (loge[0]  - loge_edge[1])
    loge_edge[-1]   = loge[-1] - (loge_edge[-2] - loge[-1])
    e_edge = 10**loge_edge

    # --- time エッジ ---
    t = Wa.time.values
    t_num = mdates.date2num(t.astype("datetime64[ms]").astype(object))
    if t_num.size == 1:
        dt = 1/24/60
        t_edge = np.array([t_num[0]-dt/2, t_num[0]+dt/2])
    else:
        t_edge = np.empty(t_num.size + 1)
        t_edge[1:-1] = 0.5*(t_num[:-1] + t_num[1:])
        t_edge[0]    = t_num[0]  - (t_edge[1]  - t_num[0])
        t_edge[-1]   = t_num[-1] + (t_num[-1] - t_edge[-2])

    Z = Wa.values

    # --- 正規化 or vlim ---
    if normalize:
        if vlim is None:
            vmax = np.nanmax(np.abs(Z)) or 1.0
            Z = Z / vmax
            vmin, vmax_plot = -1, 1
        else:
            vmax = np.nanmax(np.abs(Z)) or 1.0
            Z = Z / vmax
            vmin, vmax_plot = vlim
    else:
        if vlim is None:
            vmax_plot = np.nanmax(np.abs(Z)) or 1.0
            vmin = -vmax_plot
        else:
            vmin, vmax_plot = vlim

    pcm = ax.pcolormesh(t_edge, e_edge, Z, shading="auto",
                        cmap=cmap, vmin=vmin, vmax=vmax_plot)
    ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(True, which='both', linestyle='--', alpha=0.5)
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    if title: ax.set_title(title)
    if cax is not None:
        cb = plt.colorbar(pcm, cax=cax)
        if cbar_title: cb.set_label(cbar_title)
    return pcm

In [ ]:
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

fig = plt.figure(figsize=(10, 8), constrained_layout=False)
gs  = fig.add_gridspec(nrows=4, ncols=2, width_ratios=[1, 0.015],
                       wspace=0.025, hspace=0.1)

ax0  = fig.add_subplot(gs[0, 0])
ax1  = fig.add_subplot(gs[1, 0], sharex=ax0)
ax2  = fig.add_subplot(gs[2, 0], sharex=ax0)
ax3  = fig.add_subplot(gs[3, 0], sharex=ax0)
cax0 = fig.add_subplot(gs[0, 1])
cax2 = fig.add_subplot(gs[2, 1])
cax3 = fig.add_subplot(gs[3, 1])

ax0.tick_params(labelbottom=False)
ax1.tick_params(labelbottom=False)
ax2.tick_params(labelbottom=False)

tmin = np.datetime64('2017-11-15T16:15:00')
tmax = np.datetime64('2017-11-15T16:25:00')

tE = da_E_256Hz_perp_bandpass_amp.time.values.astype('datetime64[ms]').astype(object)
TIME, FREC = np.meshgrid(ds_spec_B_256Hz_z.time, ds_spec_B_256Hz_z.freq)
pcm = ax0.pcolormesh(TIME, FREC, ds_spec_B_256Hz_z.Sxx_Bz.T, cmap='jet', norm=LogNorm(vmin=1E-4, vmax=1E2), shading='auto')
ax0.set_ylabel(r'$B_{z}$ (DSI) Freq.' + '\n[Hz]')
ax0.minorticks_on()
ax0.set_ylim(0.4, 1.0)
ax0.grid(True, which='both', linestyle='--', alpha=0.5)
cb = plt.colorbar(pcm, cax=cax0)
cb.set_label(r'[$\mathrm{nT}^{2} / \mathrm{Hz}$]')

tB = da_B_256Hz_perp_bandpass_amp.time.values.astype('datetime64[ms]').astype(object)
ax1.plot(mdates.date2num(tB), da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax1.set_ylabel(r'$|\mathbf{B}_{\mathrm{w}}|$' + '\n[nT]')
ax1.minorticks_on()
ax1.set_ylim(0, 12)
ax1.grid(True, which='both', linestyle='--', alpha=0.5)

plot_W_on_ax(ax2, W_Eint_all, energy_range=(1., 30.), normalize=True, vlim=(-0.5, 0.5),
             time_range=(tmin, tmax), cmap='jet',
             ylabel=r'$\mathrm{H}^{+}$ energy' + '\n[keV]',
             cbar_title=r'$W_{\mathrm{Eint}}$',
             cax=cax2)

plot_W_on_ax(ax3, W_Bint_all, energy_range=(1., 30.), normalize=True, vlim=(-0.5, 0.5),
             time_range=(tmin, tmax), cmap='jet',
             ylabel=r'$\mathrm{H}^{+}$ energy' + '\n[keV]',
             cbar_title=r'$W_{\mathrm{Bint}}$',
             cax=cax3)

ax3.set_xlim(mdates.date2num([tmin, tmax]))
ax3.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

ax0.set_xlabel("time")
fig.tight_layout()
plt.show()
